In [2]:
import os, sys

os.chdir(os.path.expanduser("~/QIAO0042/models/acv/facemask/"))
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())

CWD: /scratch-share/QIAO0042/models/acv/facemask


In [3]:

import sys
sys.path.insert(0, os.path.expanduser("~/QIAO0042/models/acv/facemask/"))
import torch
from unet import UNetV2, LiteUNetV2
from palette import NUM_CLASSES

PARAM_LIMIT = 1_800_000

configs = [
    ("UNetV2",      UNetV2,      {"base": 23}),
    ("LiteUNetV2",  LiteUNetV2,  {"base": 64}),
]

print(f"{'Model':<16} {'Config':<18} {'Params':>10}  {'Status'}")
print("-" * 58)
all_ok = True
for name, cls, kwargs in configs:
    m = cls(num_classes=NUM_CLASSES, dropout=0.3, deep_supervision=True, **kwargs)
    n = sum(p.numel() for p in m.parameters())
    ok = n <= PARAM_LIMIT
    if not ok:
        all_ok = False
    status = "OK" if ok else f"EXCEEDS LIMIT by {n - PARAM_LIMIT:,}"
    cfg_str = ", ".join(f"{k}={v}" for k, v in kwargs.items())
    print(f"{name:<16} {cfg_str:<18} {n:>10,}  {status}")

print("-" * 58)
assert all_ok, "One or more models exceed the 1.8M parameter limit — fix before training!"
print(f"\nAll models are within the {PARAM_LIMIT:,} parameter limit.")


Model            Config                 Params  Status
----------------------------------------------------------
UNetV2           base=23             1,796,664  OK
LiteUNetV2       base=64             1,928,904  EXCEEDS LIMIT by 128,904
----------------------------------------------------------


AssertionError: One or more models exceed the 1.8M parameter limit — fix before training!

In [2]:
!torchrun --nproc_per_node=4 train_ddp.py

W0316 21:50:57.344000 322096 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] 
W0316 21:50:57.344000 322096 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
W0316 21:50:57.344000 322096 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0316 21:50:57.344000 322096 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
Data: /tmp/facemask/images  /tmp/facemask/masks
[rank0]:[W316 21:51:00.757511273 ProcessGroupNCCL.cpp:4115] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 to perfor

In [2]:
!torchrun --nproc_per_node=4 train_ddp.py --ckpt-path checkpoints/liteunetv2_b64.pt --epochs 150


W0316 22:23:34.635000 328675 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] 
W0316 22:23:34.635000 328675 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
W0316 22:23:34.635000 328675 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0316 22:23:34.635000 328675 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
Data: /tmp/facemask/images  /tmp/facemask/masks
[rank0]:[W316 22:23:37.031446286 ProcessGroupNCCL.cpp:4115] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 to perfor

In [3]:
!torchrun --nproc_per_node=4 train_ddp.py \
    --ckpt-path checkpoints/unetv2_testing.pt \
    --epochs 100 \
    --model unetv2 --base 23 --lr 1e-2 \
    --seed $RANDOM


W0317 22:03:16.986000 402599 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] 
W0317 22:03:16.986000 402599 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
W0317 22:03:16.986000 402599 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0317 22:03:16.986000 402599 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
Data: /tmp/facemask/images  /tmp/facemask/masks
[rank0]:[W317 22:03:20.768713328 ProcessGroupNCCL.cpp:4115] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 to perfor

In [5]:
!torchrun --nproc_per_node=4 train_ddp.py \
    --ckpt-path checkpoints/unetv2_full.pt \
    --epochs 200 \
    --model unetv2 --base 23 --lr 1e-2 \
    --full-dataset


W0317 22:13:44.806000 423183 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] 
W0317 22:13:44.806000 423183 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
W0317 22:13:44.806000 423183 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0317 22:13:44.806000 423183 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/distributed/run.py:793] *****************************************
Data: /tmp/facemask/images  /tmp/facemask/masks
[rank3]:[W317 22:13:47.410563485 ProcessGroupNCCL.cpp:4115] [PG ID 0 PG GUID 0 Rank 3]  using GPU 3 to perfor